In [33]:
%matplotlib inline
import warnings; warnings.simplefilter('ignore')

import pandas as pd
import numpy as np

df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
print("Dataset shape:", df.shape)
df.head()

In [34]:
df.info()

In [35]:
df.describe()

In [36]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())
print("\nAttrition distribution:")
print(df['Attrition'].value_counts())
print("\nAttrition percentage:")
print(df['Attrition'].value_counts(normalize=True) * 100)

In [37]:
# Separate features and target
X = df.drop('Attrition', axis=1)
y = df['Attrition']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [38]:
from sklearn.preprocessing import LabelEncoder

# Encode target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)
print("Encoded values:", np.unique(y_encoded))

In [39]:
from sklearn.model_selection import train_test_split

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Train set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nTrain target distribution:")
print(pd.Series(y_train).value_counts())
print("\nTest target distribution:")
print(pd.Series(y_test).value_counts())

In [40]:
# Feature Engineering - Create new meaningful features
def create_features(data):
    df_new = data.copy()
    
    # 1. Income-related features
    df_new['MonthlyIncome_per_Age'] = df_new['MonthlyIncome'] / (df_new['Age'] + 1)
    df_new['Income_to_HourlyRate'] = df_new['MonthlyIncome'] / (df_new['HourlyRate'] + 1)
    df_new['Income_to_DailyRate'] = df_new['MonthlyIncome'] / (df_new['DailyRate'] + 1)
    
    # 2. Work experience features
    df_new['Experience_to_CompanyTenure'] = (df_new['TotalWorkingYears'] + 1) / (df_new['YearsAtCompany'] + 1)
    df_new['Avg_Years_per_Company'] = df_new['TotalWorkingYears'] / (df_new['NumCompaniesWorked'] + 1)
    df_new['YearsCompany_to_YearsRole'] = (df_new['YearsAtCompany'] + 1) / (df_new['YearsInCurrentRole'] + 1)
    
    # 3. Career progression features
    df_new['Time_Since_Promotion'] = df_new['YearsSinceLastPromotion']
    df_new['Promotion_Rate'] = df_new['YearsAtCompany'] / (df_new['YearsSinceLastPromotion'] + 1)
    df_new['Manager_Tenure_Ratio'] = (df_new['YearsWithCurrManager'] + 1) / (df_new['YearsInCurrentRole'] + 1)
    
    # 4. Satisfaction and engagement composite features
    df_new['Overall_Satisfaction'] = (df_new['JobSatisfaction'] + df_new['EnvironmentSatisfaction'] + 
                                       df_new['JobInvolvement'] + df_new['RelationshipSatisfaction']) / 4
    df_new['Satisfaction_Variance'] = pd.DataFrame([
        df_new['JobSatisfaction'], df_new['EnvironmentSatisfaction'],
        df_new['JobInvolvement'], df_new['RelationshipSatisfaction']
    ]).var()
    df_new['WorkLife_vs_Satisfaction'] = df_new['WorkLifeBalance'] / (df_new['Overall_Satisfaction'] + 0.1)
    
    # 5. Training and development features
    df_new['Training_per_CompanyYear'] = df_new['TrainingTimesLastYear'] / (df_new['YearsAtCompany'] + 1)
    df_new['Has_Stock_Options'] = (df_new['StockOptionLevel'] > 0).astype(int)
    df_new['High_Stock_Options'] = (df_new['StockOptionLevel'] >= 2).astype(int)
    
    # 6. Demographics interaction features
    df_new['Age_DistanceFromHome'] = df_new['Age'] * df_new['DistanceFromHome']
    df_new['Senior_Employee'] = (df_new['Age'] >= 40).astype(int)
    df_new['Young_Employee'] = (df_new['Age'] < 30).astype(int)
    
    # 7. Job level and performance features
    df_new['JobLevel_Performance'] = df_new['JobLevel'] * df_new['PerformanceRating']
    df_new['SalaryHike_Performance'] = df_new['PercentSalaryHike'] * df_new['PerformanceRating']
    
    # 8. Distance and travel features
    df_new['Long_Distance_Commute'] = (df_new['DistanceFromHome'] > 10).astype(int)
    
    return df_new

# Apply feature engineering
X_train_engineered = create_features(X_train)
X_test_engineered = create_features(X_test)

print("New features created!")
print(f"Original features: {X_train.shape[1]}")
print(f"Engineered features: {X_train_engineered.shape[1]}")
print(f"New features added: {X_train_engineered.shape[1] - X_train.shape[1]}")
print("\nNew feature columns:")
new_cols = [col for col in X_train_engineered.columns if col not in X_train.columns]
for col in new_cols:
    print(f"  - {col}")

In [41]:
# One-hot encoding for categorical variables
X_train_encoded = pd.get_dummies(X_train_engineered, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_engineered, drop_first=True)

# Align test set columns with training set
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Features after one-hot encoding:")
print(f"Train shape: {X_train_encoded.shape}")
print(f"Test shape: {X_test_encoded.shape}")

In [42]:
from sklearn.preprocessing import StandardScaler

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print("Data standardized successfully!")
print(f"Training data mean: {X_train_scaled.mean():.4f}")
print(f"Training data std: {X_train_scaled.std():.4f}")
print(f"Test data mean: {X_test_scaled.mean():.4f}")
print(f"Test data std: {X_test_scaled.std():.4f}")

In [43]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("SMOTE applied successfully!")
print(f"\nBefore SMOTE:")
print(f"  Class 0: {(y_train == 0).sum()}")
print(f"  Class 1: {(y_train == 1).sum()}")
print(f"  Ratio: {(y_train == 1).sum() / (y_train == 0).sum():.2%}")

print(f"\nAfter SMOTE:")
print(f"  Class 0: {(y_train_res == 0).sum()}")
print(f"  Class 1: {(y_train_res == 1).sum()}")
print(f"  Ratio: {(y_train_res == 1).sum() / (y_train_res == 0).sum():.2%}")

In [44]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot correlation heatmap of key engineered features
plt.figure(figsize=(14, 10))
important_features = [
    'MonthlyIncome_per_Age', 'Experience_to_CompanyTenure', 'Overall_Satisfaction',
    'Promotion_Rate', 'Training_per_CompanyYear', 'WorkLife_vs_Satisfaction',
    'Has_Stock_Options', 'Age', 'MonthlyIncome', 'YearsAtCompany',
    'JobLevel', 'JobSatisfaction', 'WorkLifeBalance'
]
available_features = [f for f in important_features if f in X_train_engineered.columns]
sns.heatmap(X_train_engineered[available_features].corr(), cmap='coolwarm', annot=False, fmt='.2f')
plt.title("Correlation Heatmap - Engineered Features")
plt.tight_layout()
plt.show()

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Grid search for Logistic Regression with resampled data
lr = LogisticRegression(max_iter=1000)

param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'class_weight': ['balanced', None]
}

grid_lr = GridSearchCV(
    lr, param_grid_lr, cv=5, scoring='recall', n_jobs=-1, verbose=1
)
grid_lr.fit(X_train_res, y_train_res)

print("\nBest Logistic Regression Parameters:")
print(grid_lr.best_params_)
print(f"Best Cross-Validation Recall Score: {grid_lr.best_score_:.4f}")

In [46]:
from sklearn.svm import SVC

# Grid search for SVM with resampled data
svm = SVC(probability=True)

param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

grid_svm = GridSearchCV(
    svm, param_grid_svm, cv=5, scoring='recall', n_jobs=-1, verbose=1
)
grid_svm.fit(X_train_res, y_train_res)

print("\nBest SVM Parameters:")
print(grid_svm.best_params_)
print(f"Best Cross-Validation Recall Score: {grid_svm.best_score_:.4f}")

In [47]:
from sklearn.tree import DecisionTreeClassifier

# Grid search for Decision Tree
dt = DecisionTreeClassifier(random_state=42)

param_grid_dt = {
    'max_depth': [3, 4, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None]
}

grid_dt = GridSearchCV(
    dt, param_grid_dt, cv=5, scoring='recall', n_jobs=-1, verbose=1
)
grid_dt.fit(X_train_encoded, y_train)

print("\nBest Decision Tree Parameters:")
print(grid_dt.best_params_)
print(f"Best Cross-Validation Recall Score: {grid_dt.best_score_:.4f}")

In [48]:
# Get best estimators
best_lr = grid_lr.best_estimator_
best_svm = grid_svm.best_estimator_
best_dt = grid_dt.best_estimator_

print("Best models selected!")

In [49]:
# Make predictions
y_pred_lr = best_lr.predict(X_test_scaled)
y_pred_svm = best_svm.predict(X_test_scaled)
y_pred_dt = best_dt.predict(X_test_encoded)

print("Predictions generated for all models!")

In [50]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("="*70)
print("LOGISTIC REGRESSION - CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred_lr))
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

In [51]:
print("="*70)
print("SVM - CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred_svm))
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

In [52]:
print("="*70)
print("DECISION TREE - CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred_dt))
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

In [53]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve

# Get probability predictions for AUC calculations
y_pred_proba_lr = best_lr.predict_proba(X_test_scaled)[:, 1]
y_pred_proba_svm = best_svm.predict_proba(X_test_scaled)[:, 1]
y_pred_proba_dt = best_dt.predict_proba(X_test_encoded)[:, 1]

# Calculate AUC-ROC scores
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
auc_svm = roc_auc_score(y_test, y_pred_proba_svm)
auc_dt = roc_auc_score(y_test, y_pred_proba_dt)

print("="*70)
print("MODEL COMPARISON - AUC-ROC SCORES")
print("="*70)
print(f"Logistic Regression AUC: {auc_lr:.4f}")
print(f"SVM AUC: {auc_svm:.4f}")
print(f"Decision Tree AUC: {auc_dt:.4f}")
print("\nBest Model: ", end="")
best_model = max([("Logistic Regression", auc_lr), ("SVM", auc_svm), ("Decision Tree", auc_dt)], key=lambda x: x[1])
print(f"{best_model[0]} (AUC: {best_model[1]:.4f})")

In [54]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 8))

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_pred_proba_svm)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_pred_proba_dt)

plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={auc_lr:.3f})', linewidth=2)
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC={auc_svm:.3f})', linewidth=2)
plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC={auc_dt:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [55]:
# Feature importance from Decision Tree
feature_importance = pd.DataFrame({
    'feature': X_train_encoded.columns,
    'importance': best_dt.feature_importances_
}).sort_values('importance', ascending=False)

print("="*70)
print("TOP 20 MOST IMPORTANT FEATURES (Decision Tree)")
print("="*70)
print(feature_importance.head(20).to_string(index=False))

# Visualize top features
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
plt.barh(top_features['feature'], top_features['importance'], color='steelblue')
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 20 Most Important Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [56]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("="*70)
print("DETAILED MODEL PERFORMANCE METRICS")
print("="*70)

models = {
    'Logistic Regression': y_pred_lr,
    'SVM': y_pred_svm,
    'Decision Tree': y_pred_dt
}

metrics_data = []
for model_name, y_pred in models.items():
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)
    
    if model_name == 'Logistic Regression':
        auc = auc_lr
    elif model_name == 'SVM':
        auc = auc_svm
    else:
        auc = auc_dt
    
    metrics_data.append({
        'Model': model_name,
        'Accuracy': f'{accuracy:.4f}',
        'Precision': f'{precision:.4f}',
        'Recall': f'{recall:.4f}',
        'F1-Score': f'{f1:.4f}',
        'AUC-ROC': f'{auc:.4f}'
    })

metrics_df = pd.DataFrame(metrics_data)
print(metrics_df.to_string(index=False))

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Total Samples: {len(y_test)}")
print(f"Positive Class (Attrition): {(y_test == 1).sum()} ({(y_test == 1).sum()/len(y_test)*100:.1f}%)")
print(f"Negative Class (No Attrition): {(y_test == 0).sum()} ({(y_test == 0).sum()/len(y_test)*100:.1f}%)")
print("\nFeature Engineering: 21 new features created")
print("Data Balancing: SMOTE applied to training data")
print(f"Final Feature Count: {X_train_encoded.shape[1]}")